In [ ]:
# HCC Recurrence Prediction - Training and Evaluation Notebook
# Complete pipeline with SHAP explanations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import joblib
import shap
import warnings
import os

warnings.filterwarnings('ignore')

# Create directories if they don't exist
os.makedirs('../models/trained_models', exist_ok=True)
os.makedirs('../models/preprocessing', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load the data
print("Loading HCC Dataset...")
df = pd.read_excel('../data/raw/hcc-data-complete-balanced.xlsx')


print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Data preprocessing and feature engineering
print("\n🔧 Feature Engineering...")

# Create a copy for feature engineering
df_processed = df.copy()

# 1. Create Nodule feature from Nodules
if 'Nodules' in df_processed.columns:
    df_processed['Nodule'] = df_processed['Nodules']

# 2. Age categorization
if 'Age' in df_processed.columns:
    age_vals = df_processed['Age'].values
    age_cat = np.zeros_like(age_vals, dtype=float)
    age_cat[(age_vals >= 50) & (age_vals <= 65)] = 1.0
    age_cat[age_vals > 65] = 2.0
    df_processed['Age_Category'] = age_cat

# 3. AFP risk categorization
if 'AFP' in df_processed.columns:
    afp_vals = df_processed['AFP'].values
    afp_cat = np.zeros_like(afp_vals, dtype=float)
    afp_cat[(afp_vals >= 20) & (afp_vals <= 400)] = 1.0
    afp_cat[afp_vals > 400] = 2.0
    df_processed['AFP_Risk_Category'] = afp_cat

# 4. Liver function composite score
liver_markers = ['Albumin', 'Total_Bil', 'INR', 'ALT', 'AST']
available_markers = [m for m in liver_markers if m in df_processed.columns]

if len(available_markers) >= 2:
    scores = []
    for marker in available_markers:
        val = df_processed[marker]
        if marker == 'Albumin':
            # Lower albumin is worse
            s = 1.0 - (val / 5.5)  # Normalize to 0-1 range
        else:
            # Higher values are worse
            s = val / 100.0  # Rough normalization
        scores.append(s.fillna(0))
    
    df_processed['Liver_Function_Score'] = pd.concat(scores, axis=1).mean(axis=1)
else:
    df_processed['Liver_Function_Score'] = 0.0

# Define features and target
print("\n Defining features and target...")

# Expected features based on the dataset
expected_features = [
    'Obesity', 'Hallmark', 'HBeAg', 'Ferritin', 'CRI', 'Diabetes', 'TP', 
    'Encephalopathy', 'PVT', 'PS', 'INR', 'Hemoglobin', 'Platelets', 
    'Alcohol', 'Age', 'Total_Bil', 'Ascites', 'ALP', 'ALT', 'Symptoms', 
    'Gender', 'HIV', 'Endemic', 'Sat', 'Smoking', 'HBcAb', 'Grams_day', 
    'AFP', 'Nodule', 'Spleno', 'PHT', 'Iron', 'Cirrhosis', 'Albumin', 
    'Major_Dim', 'AHT', 'Varices', 'Hemochro', 'HBsAg', 'HCVAb', 'GGT', 
    'MCV', 'Dir_Bil', 'AST', 'Metastasis', 'Packs_year', 'Creatinine', 
    'NASH', 'Leucocytes', 'Age_Category', 'AFP_Risk_Category', 'Liver_Function_Score'
]

# Ensure all expected features are present
for feature in expected_features:
    if feature not in df_processed.columns:
        df_processed[feature] = 0.0

# Select only the expected features
X = df_processed[expected_features]
y = df['Recurrence'] if 'Recurrence' in df.columns else df['Class']  # Adjust based on your dataset

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

# Train-test split
print("\n  Creating train-test split...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Define preprocessing pipeline
print("\n  Creating preprocessing pipeline...")

# Identify numeric columns (all columns in our case are numeric)
numeric_features = X.columns.tolist()

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Create full pipeline with classifier
def create_model_pipeline(classifier):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])

# Define models to evaluate
models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ),
    'LogisticRegression': LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42
    )
}

# Train and evaluate models
print("\n Training and evaluating models...")
results = {}

for name, model in models.items():
    print(f"\n Evaluating {name}...")
    
    pipeline = create_model_pipeline(model)
    
    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, 
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                                scoring='roc_auc')
    
    print(f"  CV ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
    
    # Train on full training set
    pipeline.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    test_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'pipeline': pipeline,
        'cv_score': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_auc': test_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  Test ROC-AUC: {test_auc:.3f}")

# Select best model
best_model_name = max(results.keys(), key=lambda x: results[x]['test_auc'])
best_model = results[best_model_name]
print(f"\n Best model: {best_model_name}")
print(f"   Test ROC-AUC: {best_model['test_auc']:.3f}")

# Save the best model
print("\n Saving best model and preprocessor...")
model_dir = '../models'
os.makedirs(f'{model_dir}/trained_models', exist_ok=True)
os.makedirs(f'{model_dir}/preprocessing', exist_ok=True)

# Save the full pipeline
joblib.dump(best_model['pipeline'], f'{model_dir}/trained_models/best_model.pkl')

# Save preprocessor separately for use in Flask app
preprocessor_data = {
    'pipeline': preprocessor,
    'feature_names': X.columns.tolist()
}
joblib.dump(preprocessor_data, f'{model_dir}/preprocessing/preprocessor.pkl')

print(" Model saved successfully!")

# SHAP Analysis with fixed expected value extraction
print("\n Performing SHAP Analysis...")

# Helper function to extract scalar expected value
def extract_shap_expected_value(explainer):
    """Extract scalar expected value from SHAP explainer."""
    expected_value = explainer.expected_value
    
    if isinstance(expected_value, np.ndarray):
        if expected_value.size == 2:
            # For binary classification, use positive class
            return expected_value[1]
        else:
            return expected_value[0]
    elif isinstance(expected_value, list):
        if len(expected_value) == 2:
            return expected_value[1]
        else:
            return expected_value[0]
    else:
        return expected_value

# Helper function to extract SHAP values for positive class
def extract_shap_values_positive(shap_values):
    """Extract SHAP values for positive class from various formats."""
    if isinstance(shap_values, list):
        # Binary classification - shap_values is [negative_class, positive_class]
        if len(shap_values) == 2:
            return shap_values[1]  # Positive class (recurrence)
        else:
            return shap_values[0]
    elif hasattr(shap_values, 'shape'):
        if len(shap_values.shape) == 3:
            # Shape is (n_samples, n_features, n_classes)
            if shap_values.shape[2] >= 2:
                return shap_values[:, :, 1]  # Class 1
            else:
                return shap_values[:, :, 0]
        elif len(shap_values.shape) == 2:
            return shap_values
        else:
            # Reshape 1D array to 2D
            return shap_values.reshape(-1, 1)
    else:
        return shap_values

# Get the classifier from the pipeline
classifier = best_model['pipeline'].named_steps['classifier']
preprocessor_only = best_model['pipeline'].named_steps['preprocessor']

# Transform the data
X_train_processed = preprocessor_only.transform(X_train)
X_test_processed = preprocessor_only.transform(X_test)

# Initialize SHAP explainer
if hasattr(classifier, 'feature_importances_'):
    # Tree-based models
    print(f" Initializing SHAP TreeExplainer for {best_model_name}...")
    explainer = shap.TreeExplainer(classifier)
    shap_values = explainer.shap_values(X_test_processed)
    
    # Extract scalar expected value
    expected_value_scalar = extract_shap_expected_value(explainer)
    print(f" SHAP expected value (scalar): {expected_value_scalar:.4f}")
    
    # Extract SHAP values for positive class
    shap_values_positive = extract_shap_values_positive(shap_values)
    print(f" SHAP values shape: {shap_values_positive.shape}")
    print(f" X_test_processed shape: {X_test_processed.shape}")
    
    # Ensure shapes match
    if len(shap_values_positive.shape) == 1:
        shap_values_positive = shap_values_positive.reshape(1, -1)
    
    if shap_values_positive.shape[1] != X_test_processed.shape[1]:
        print(f" Shape mismatch! SHAP: {shap_values_positive.shape[1]}, Features: {X_test_processed.shape[1]}")
        # Take only the matching number of features
        min_features = min(shap_values_positive.shape[1], X_test_processed.shape[1])
        shap_values_positive = shap_values_positive[:, :min_features]
        X_test_for_shap = X_test_processed[:, :min_features]
        feature_names_for_shap = X.columns.tolist()[:min_features]
    else:
        X_test_for_shap = X_test_processed
        feature_names_for_shap = X.columns.tolist()
    
    # 1. Summary plot
    print(" Creating SHAP summary plot...")
    try:
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values_positive, X_test_for_shap, 
                         feature_names=feature_names_for_shap, show=False)
        plt.title(f'SHAP Summary Plot - {best_model_name}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../reports/shap_summary_plot.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(" Summary plot created successfully")
    except Exception as e:
        print(f" Could not create summary plot: {e}")
    
    # 2. Bar plot (mean absolute SHAP values)
    print(" Creating SHAP bar plot...")
    try:
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values_positive, X_test_for_shap, 
                         feature_names=feature_names_for_shap, plot_type="bar", show=False)
        plt.title(f'Feature Importance (SHAP) - {best_model_name}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../reports/shap_bar_plot.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(" Bar plot created successfully")
    except Exception as e:
        print(f" Could not create bar plot: {e}")
    
    # 3. Feature importance from SHAP values
    print(" Creating SHAP feature importance plot...")
    try:
        # Calculate mean absolute SHAP values
        mean_abs_shap = np.abs(shap_values_positive).mean(axis=0)
        
        # Create DataFrame for plotting
        shap_importance = pd.DataFrame({
            'feature': feature_names_for_shap,
            'importance': mean_abs_shap
        }).sort_values('importance', ascending=False).head(15)
        
        # Plot
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(shap_importance)), shap_importance['importance'][::-1])
        plt.yticks(range(len(shap_importance)), shap_importance['feature'][::-1])
        plt.xlabel('Mean |SHAP Value|', fontsize=12)
        plt.title(f'Top 15 Features by SHAP Importance - {best_model_name}', fontsize=16, fontweight='bold')
        plt.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.savefig('../reports/shap_feature_importance.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(" SHAP feature importance plot created successfully")
    except Exception as e:
        print(f" Could not create SHAP feature importance plot: {e}")
    
    # 4. Waterfall plots for sample patients (FIXED VERSION)
    print(" Generating SHAP Waterfall Plots for Sample Patients...")
    
    # Select sample patients from different risk groups
    y_test_proba = best_model['pipeline'].predict_proba(X_test)[:, 1]
    
    # Find indices for low, medium, high risk
    low_risk_idx = np.where(y_test_proba < 0.3)[0]
    medium_risk_idx = np.where((y_test_proba >= 0.3) & (y_test_proba < 0.7))[0]
    high_risk_idx = np.where(y_test_proba >= 0.7)[0]
    
    sample_indices = []
    sample_labels = []
    
    if len(low_risk_idx) > 0:
        sample_indices.append(low_risk_idx[0])
        sample_labels.append("Low")
    if len(medium_risk_idx) > 0:
        sample_indices.append(medium_risk_idx[len(medium_risk_idx)//2])
        sample_labels.append("Medium")
    if len(high_risk_idx) > 0:
        sample_indices.append(high_risk_idx[-1])
        sample_labels.append("High")
    
    print(f" Sample patients selected:")
    for idx, label in zip(sample_indices, sample_labels):
        prob = y_test_proba[idx]
        print(f"  • Patient {idx}: {label} risk ({prob*100:.1f}%)")
    
    # Create waterfall plots using the fixed scalar expected value
    for idx, label in zip(sample_indices, sample_labels):
        prob = y_test_proba[idx]
        
        try:
            # Get SHAP values for this specific sample
            if len(shap_values_positive.shape) == 2:
                sample_shap_values = shap_values_positive[idx]
            else:
                sample_shap_values = shap_values_positive
            
            # Ensure we have the right data for this sample
            sample_features = X_test_for_shap[idx]
            
            # Create waterfall plot with FIXED expected value
            plt.figure(figsize=(14, 8))
            
            # Create SHAP Explanation object
            explanation = shap.Explanation(
                values=sample_shap_values,
                base_values=expected_value_scalar,
                data=sample_features,
                feature_names=feature_names_for_shap
            )
            
            # Create waterfall plot
            shap.plots.waterfall(explanation, max_display=15, show=False)
            
            plt.title(f'SHAP Waterfall Plot - Patient {idx} ({label} Risk, {prob*100:.1f}%)', 
                     fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'../reports/waterfall_patient_{idx}.png', dpi=150, bbox_inches='tight')
            plt.show()
            print(f" Waterfall plot created for patient {idx}")
            
        except Exception as e:
            print(f" Could not create waterfall plot for patient {idx}: {e}")
            # Alternative: Create simple bar plot
            try:
                plt.figure(figsize=(12, 8))
                
                # Get top 10 features by absolute SHAP value for this patient
                abs_shap = np.abs(sample_shap_values)
                top_n = min(10, len(abs_shap))
                top_idx = np.argsort(abs_shap)[-top_n:][::-1]
                
                # Create horizontal bar plot
                y_pos = np.arange(top_n) 
                colors = ['#ff6b6b' if sample_shap_values[i] > 0 else '#51cf66' for i in top_idx]
                
                plt.barh(y_pos, sample_shap_values[top_idx], color=colors, edgecolor='black', height=0.7)
                plt.yticks(y_pos, [feature_names_for_shap[i] for i in top_idx])
                plt.xlabel('SHAP Value (Impact on Prediction)', fontsize=12)
                plt.title(f'Top Features - Patient {idx} ({label} Risk, {prob*100:.1f}%)', 
                         fontsize=14, fontweight='bold')
                plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
                
                # Add value labels
                for i, (pos, val) in enumerate(zip(y_pos, sample_shap_values[top_idx])):
                    plt.text(val if abs(val) > 0.01 else 0.01, pos, 
                            f'{val:.3f}', va='center', 
                            ha='left' if val >= 0 else 'right',
                            fontsize=9)
                
                plt.grid(True, alpha=0.3, axis='x')
                plt.tight_layout()
                plt.savefig(f'../reports/bar_patient_{idx}.png', dpi=150, bbox_inches='tight')
                plt.show()
                print(f" Created bar plot for patient {idx}")
            except Exception as e2:
                print(f" Could not create bar plot either: {e2}")
    
    print(" SHAP Dependence Plots saved to '../reports/shap_dependence_plots.png'")
    
else:
    print(" SHAP analysis not supported for this model type")

# Model performance visualization
print("\n Creating Model Performance Visualizations...")

# 1. ROC Curve
plt.figure(figsize=(10, 8))
for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=16, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Confusion Matrix for best model
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, best_model['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Recurrence', 'Recurrence'],
            yticklabels=['No Recurrence', 'Recurrence'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Feature Importance for tree-based models
if hasattr(classifier, 'feature_importances_'):
    plt.figure(figsize=(12, 8))
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': classifier.feature_importances_
    }).sort_values('importance', ascending=False).head(15)
    
    plt.barh(range(len(feature_importance)), feature_importance['importance'][::-1])
    plt.yticks(range(len(feature_importance)), feature_importance['feature'][::-1])
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top 15 Feature Importance - {best_model_name}', fontsize=16, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

# 4. Probability Distribution
plt.figure(figsize=(10, 6))
for name, result in results.items():
    if len(result['y_pred_proba'][y_test == 0]) > 0:
        sns.kdeplot(result['y_pred_proba'][y_test == 0], label=f'{name} - No Recurrence', alpha=0.6)
    if len(result['y_pred_proba'][y_test == 1]) > 0:
        sns.kdeplot(result['y_pred_proba'][y_test == 1], label=f'{name} - Recurrence', alpha=0.6)

plt.xlabel('Predicted Probability', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Predicted Probability Distributions', fontsize=16, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/probability_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Create comprehensive evaluation report
print("\n Generating Evaluation Report...")

report = f"""
HCC RECURRENCE PREDICTION MODEL EVALUATION REPORT
===================================================

Dataset Information:
-------------------
- Total samples: {df.shape[0]}
- Features: {df.shape[1]}
- Train samples: {X_train.shape[0]}
- Test samples: {X_test.shape[0]}

Target Distribution:
-------------------
{y.value_counts().to_string()}

Model Performance:
-----------------
"""

for name, result in results.items():
    report += f"""
{name}:
  - Cross-Validation ROC-AUC: {result['cv_score']:.3f} (+/- {result['cv_std']:.3f})
  - Test ROC-AUC: {result['test_auc']:.3f}
"""

report += f"""
Best Model: {best_model_name}
- Test ROC-AUC: {best_model['test_auc']:.3f}
- Cross-Validation Score: {best_model['cv_score']:.3f}

SHAP Analysis:
-------------
- Expected value (baseline): {expected_value_scalar if 'expected_value_scalar' in locals() else 'N/A':.4f}
- Top features analyzed using SHAP values
- Visualizations saved to '../reports/' directory

Files Saved:
-----------
1. ../models/trained_models/best_model.pkl
2. ../models/preprocessing/preprocessor.pkl
3. ../reports/shap_summary_plot.png
4. ../reports/shap_bar_plot.png
5. ../reports/shap_feature_importance.png
6. ../reports/roc_curves.png
7. ../reports/confusion_matrix.png
8. ../reports/feature_importance.png
9. ../reports/probability_distributions.png

Next Steps:
----------
1. The model is ready for deployment in the Flask web application
2. Use the saved model files in the webapp directory
3. Monitor model performance on new data
4. Retrain periodically with updated data
"""

print(report)

# Save the report
with open('../reports/model_evaluation_report.txt', 'w') as f:
    f.write(report)

print(" Report saved to '../reports/model_evaluation_report.txt'")

# Final summary
print("\n" + "="*60)
print(" MODEL TRAINING COMPLETE!")
print("="*60)
print(f" Best Model: {best_model_name}")
print(f" Test ROC-AUC: {best_model['test_auc']:.3f}")
print(f" Model saved to: ../models/trained_models/best_model.pkl")
print(f" Visualizations saved to: ../reports/")
print("="*60)
print("\n Ready for deployment to Flask web application!")